In [1]:
import re
import numpy as np
import pandas as pd
import joblib
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
# ========== 1. Load Data ==========
file_path = "data/csv/Data_tanggapan_positif.xlsx"
df_krisis = pd.read_excel(file_path, sheet_name="Krisis")
df_tidak = pd.read_excel(file_path, sheet_name="Tidak Krisis")

df_krisis = df_krisis.rename(columns={"Kalimat": "text", "Respon": "tanggapan"})
df_tidak = df_tidak.rename(columns={"Kalimat": "text", "Respon": "tanggapan"})

df_krisis["label"] = 1
df_tidak["label"] = 0
df = pd.concat([df_krisis, df_tidak], ignore_index=True)
df = df.dropna(subset=["text", "tanggapan"])

def preprocess_text(s):
    s = str(s).lower()
    s = re.sub(r'[^0-9a-z\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

df["text_proc"] = df["text"].apply(preprocess_text)

In [3]:
# ========== 2. Split ==========
X = df["text_proc"]
y = df["label"]
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1, stratify=y_train_full, random_state=42)

In [4]:
# ========== 3. Vectorizer + Model ==========
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=10000, sublinear_tf=True, min_df=2, max_df=0.95)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

model = LGBMClassifier(
    class_weight='balanced',
    n_estimators=300,
    learning_rate=1.618e-2,
    max_depth=7,
    random_state=42
)
model.fit(X_train_tfidf, y_train)

[LightGBM] [Info] Number of positive: 673, number of negative: 720
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001917 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2995
[LightGBM] [Info] Number of data points in the train set: 1393, number of used features: 136
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

LGBMClassifier(class_weight='balanced', learning_rate=0.01618, max_depth=7,
               n_estimators=300, random_state=42)

In [5]:
# ========== 4. Threshold Tuning ==========
probs_val = model.predict_proba(X_val_tfidf)[:, 1]
best = {"thresh": 0.5, "recall": 0.0, "precision": 0.0, "f1": 0.0}
for t in np.linspace(0.1, 0.95, 85):
    preds = (probs_val >= t).astype(int)
    r = recall_score(y_val, preds)
    p = precision_score(y_val, preds, zero_division=0)
    f = f1_score(y_val, preds, zero_division=0)
    if r > best["recall"] - 1e-9 and (r > best["recall"] or p > best["precision"] * 0.7):
        best = {"thresh": t, "recall": r, "precision": p, "f1": f}

chosen_threshold = best["thresh"]
print(f"✅ Threshold dipilih: {chosen_threshold:.3f} | recall={best['recall']:.3f}, prec={best['precision']:.3f}, f1={best['f1']:.3f}")

d:\Projects\Robot Pencegah Bunuh Diri\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ Threshold dipilih: 0.323 | recall=1.000, prec=0.987, f1=0.993


In [6]:
# ========== 5. Save Model ==========
import os
os.makedirs("models/language", exist_ok=True)

joblib.dump(vectorizer, "models/language/vectorizer.pkl")
joblib.dump(model, "models/language/lgbm_model.pkl")
with open("models/language/threshold.txt", "w") as f:
    f.write(str(chosen_threshold))

print("✅ Model & threshold saved to /models/language/")

✅ Model & threshold saved to /models/language/
